# 09. NLR M3+R3 workload generator 및 20 MW IT aggregation

선정된 M3 정상상태 backbone과 R3 AR(1)+event residual을 portable generator로 저장하고, 비동기 workload diversity와 common-mode 동기화를 함께 고려하여 평균 20 MW의 facility IT 부하 프로파일을 생성한다. `n_effective`는 실제 서버 대수가 아니라 통계적 유효 단위 수이며, `common_mode_fraction`은 현재 NLR 자료만으로 식별할 수 없으므로 민감도 변수로 취급한다.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
from IPython.display import Image, display
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
OUTPUT = PROJECT_ROOT / 'outputs' / 'workload_aggregation'
ARTIFACT = OUTPUT / 'workload_generator_artifact'

In [ ]:
completed = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts' / 'run_workload_aggregation.py'),
     '--target-it-mw', '20', '--draws', '10'],
    cwd=PROJECT_ROOT, check=True, text=True, capture_output=True
)
print(completed.stdout)

## Aggregation 가정

집합 residual은 $e_{agg}=\sqrt{\rho}e_{common}+\sqrt{1-\rho}\,N_{eff}^{-1}\sum_i e_i$로 구성한다. 따라서 동일 분산의 독립 residual을 가정하면 이론적 표준편차 비율은 $\sqrt{\rho+(1-\rho)/N_{eff}}$이다. $\rho=0$은 독립 변동의 평균화, $\rho=1$은 하나의 profile을 전체 설비에 동기적으로 확대하는 경우에 대응한다.

In [ ]:
decision = json.loads((OUTPUT / 'aggregation_methodology_decision.json').read_text(encoding='utf-8'))
manifest = json.loads((ARTIFACT / 'artifact_manifest.json').read_text(encoding='utf-8'))
display(decision)
display(manifest)

In [ ]:
summary = pd.read_csv(OUTPUT / 'aggregation_summary.csv')
representative = summary[
    (summary['request_rate'] == 90)
    & (summary['n_effective'].isin([1, 50, 200]))
    & (summary['common_mode_fraction'].isin([0.0, 0.25, 1.0]))
][[
    'prompt_dataset', 'n_effective', 'common_mode_fraction',
    'theoretical_diversity_factor', 'coefficient_of_variation_mean',
    'p95_abs_ramp_mw_per_s_mean', 'minimum_it_power_mw_mean',
    'maximum_it_power_mw_mean'
]]
display(representative.reset_index(drop=True))

In [ ]:
for name in [
    'diversity_cv_curves.png',
    'diversity_ramp_curves.png',
    'representative_20mw_profiles.png',
]:
    display(Image(filename=str(OUTPUT / 'figures' / name)))

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from nlr_data.workload_generator import WorkloadGenerator
generator = WorkloadGenerator.from_artifact(ARTIFACT)
example = generator.generate_facility_profile(
    prompt_dataset='InstructCoder', request_rate=90,
    target_it_mean_mw=20, n_effective=50,
    common_mode_fraction=0.25, seed=42
)
print(f"samples={len(example)}, mean={example['it_power_mw'].mean():.12f} MW")
display(example.head())

In [ ]:
print((OUTPUT / 'workload_aggregation_report_ko.md').read_text(encoding='utf-8'))